# Servidor de keypoints para deteccion de caidas (Google Colab)

Version para quienes no tienen RunPod. Hace lo mismo que el notebook de RunPod: recibe imagenes JPEG
y devuelve, por cada persona, su caja y sus 17 puntos clave COCO. El cliente `cliente_local.py`
no cambia; solo usa la URL publica que genera este notebook.

**Antes de empezar:** *Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU (T4)*.

**Modelo por defecto:** Detectron2, Keypoint R-CNN R50-FPN 3x.
**Alternativa comentada:** YOLO-pose de Ultralytics (marcas `ALTERNAR` en las celdas 2 y 4).

**Exposicion a internet:** Colab no tiene puertos publicos, asi que se usa un tunel rapido de
Cloudflare (`trycloudflare.com`), que no requiere cuenta y soporta WebSocket. Como alternativa se
deja ngrok comentado (requiere cuenta gratuita y authtoken).

Limitaciones de Colab:
- La URL del tunel cambia cada vez que se ejecuta el notebook.
- La sesion se cierra por inactividad o al alcanzar el limite de uso; hay que volver a ejecutar todo.
- Esta pensado para practicas en clase, no para monitoreo continuo.

## 1. Configuracion

In [ ]:
import os, secrets

PUERTO = 8000

# Token que el cliente debe enviar. Se genera uno nuevo en cada ejecucion; puedes fijarlo a mano.
API_TOKEN = secrets.token_urlsafe(16)
# API_TOKEN = "mi_token_fijo"

# Umbral de confianza para aceptar una persona detectada
UMBRAL_PERSONA = 0.6

# Tunel: "cloudflare" (sin cuenta) o "ngrok" (requiere authtoken)
TUNEL = "cloudflare"
NGROK_AUTHTOKEN = ""   # solo si TUNEL = "ngrok": https://dashboard.ngrok.com/get-started/your-authtoken

print("Token:", API_TOKEN)

## 2. Instalacion de dependencias

Detectron2 se compila contra el PyTorch que trae Colab; tarda varios minutos. Si la compilacion falla
(Colab actualiza su PyTorch con frecuencia), usa la alternativa YOLO-pose.

Si al ejecutar la celda 5 aparece un `ImportError` relacionado con `typing_extensions`, reinicia la
sesion (*Entorno de ejecucion > Reiniciar sesion*) y ejecuta de nuevo desde la celda 1, saltando esta.

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available(),
      "| CUDA build:", torch.version.cuda)

!pip install -q fastapi "uvicorn[standard]" opencv-python-headless ninja "typing_extensions>=4.14"

# --- Detectron2 (por defecto) ---
!pip install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"

# --- ALTERNAR: YOLO-pose (descomenta si vas a usar YOLO; puedes comentar la linea de Detectron2) ---
# !pip install -q ultralytics

# --- Tunel Cloudflare ---
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

# --- ALTERNAR: tunel ngrok ---
# !pip install -q pyngrok

## 3. Verificacion del entorno

In [ ]:
import numpy as np, cv2
print("OpenCV:", cv2.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Sin GPU: la inferencia funcionara en CPU, bastante mas lenta.")

try:
    import detectron2
    print("Detectron2:", detectron2.__version__)
except ImportError:
    print("Detectron2 no instalado (correcto solo si vas a usar YOLO-pose).")

## 4. Carga del modelo

Ambas clases devuelven exactamente el mismo formato:

```
[{"caja": [x1, y1, x2, y2], "score": 0.98, "keypoints": [[x, y, conf], ... 17 puntos]}]
```

Nota sobre Detectron2: el tercer valor de `pred_keypoints` es el maximo del mapa de calor (un logit,
no una probabilidad). Se pasa por una sigmoide para dejarlo en [0, 1] y que el cliente use el mismo
umbral (`--conf-kp 0.5`) con cualquiera de los dos modelos.

In [ ]:
import time

def _sigmoide(x):
    return 1.0 / (1.0 + np.exp(-x))


# ===================== Detectron2: Keypoint R-CNN (por defecto) =====================
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

class PoseDetectron2:
    CONFIG = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
    # Variante mas precisa y mas lenta: "COCO-Keypoints/keypoint_rcnn_X_101_32x8d_FPN_3x.yaml"

    def __init__(self, umbral=0.6):
        cfg = get_cfg()
        cfg.merge_from_file(model_zoo.get_config_file(self.CONFIG))
        cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(self.CONFIG)
        cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = umbral
        cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        self.predictor = DefaultPredictor(cfg)  # espera imagenes BGR, como OpenCV
        self.nombre = "detectron2/" + self.CONFIG.split("/")[-1].replace(".yaml", "")
        self.dispositivo = cfg.MODEL.DEVICE

    def __call__(self, img_bgr):
        inst = self.predictor(img_bgr)["instances"].to("cpu")
        if len(inst) == 0:
            return []
        cajas = inst.pred_boxes.tensor.numpy()
        scores = inst.scores.numpy()
        kps = inst.pred_keypoints.numpy().copy()     # (N, 17, 3)
        kps[:, :, 2] = _sigmoide(kps[:, :, 2])
        return [{"caja": [round(float(v), 1) for v in cajas[i]],
                 "score": round(float(scores[i]), 3),
                 "keypoints": [[round(float(x), 1), round(float(y), 1), round(float(c), 3)]
                               for x, y, c in kps[i]]}
                for i in range(len(inst))]


# ===================== ALTERNAR: YOLO-pose de Ultralytics =====================
# from ultralytics import YOLO
#
# class PoseYOLO:
#     # Tamanos: yolo11n-pose (rapido) ... yolo11x-pose (preciso). Se descargan solos.
#     def __init__(self, umbral=0.6, pesos="yolo11n-pose.pt"):
#         self.modelo = YOLO(pesos)
#         self.umbral = umbral
#         self.dispositivo = 0 if torch.cuda.is_available() else "cpu"
#         self.nombre = "ultralytics/" + pesos.replace(".pt", "")
#
#     def __call__(self, img_bgr):
#         r = self.modelo.predict(img_bgr, conf=self.umbral, classes=[0],
#                                 device=self.dispositivo, verbose=False)[0]
#         if r.keypoints is None or len(r.boxes) == 0:
#             return []
#         cajas = r.boxes.xyxy.cpu().numpy()
#         scores = r.boxes.conf.cpu().numpy()
#         xy = r.keypoints.xy.cpu().numpy()                     # (N, 17, 2)
#         conf = (r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None
#                 else np.ones(xy.shape[:2]))                   # (N, 17)
#         return [{"caja": [round(float(v), 1) for v in cajas[i]],
#                  "score": round(float(scores[i]), 3),
#                  "keypoints": [[round(float(xy[i, k, 0]), 1), round(float(xy[i, k, 1]), 1),
#                                 round(float(conf[i, k]), 3)] for k in range(xy.shape[1])]}
#                 for i in range(len(cajas))]


# ===================== Seleccion del modelo activo =====================
modelo = PoseDetectron2(umbral=UMBRAL_PERSONA)
# modelo = PoseYOLO(umbral=UMBRAL_PERSONA)      # ALTERNAR: descomenta esta y comenta la anterior

# Calentamiento: la primera inferencia siempre es mas lenta (reserva de memoria, kernels CUDA).
_ = modelo(np.zeros((480, 640, 3), dtype=np.uint8))
t0 = time.perf_counter()
for _i in range(5):
    modelo(np.zeros((480, 640, 3), dtype=np.uint8))
print(f"Modelo: {modelo.nombre} en {modelo.dispositivo} | "
      f"{(time.perf_counter() - t0) / 5 * 1000:.0f} ms por imagen 640x480")

## 5. Definicion de la API

In [ ]:
import json, threading
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Request, Header, HTTPException
from starlette.concurrency import run_in_threadpool

app = FastAPI(title="API de keypoints para deteccion de caidas")
_lock_modelo = threading.Lock()   # una inferencia a la vez sobre la GPU


def inferir_jpeg(datos: bytes) -> dict:
    img = cv2.imdecode(np.frombuffer(datos, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("No se pudo decodificar la imagen")
    t0 = time.perf_counter()
    with _lock_modelo:
        personas = modelo(img)
    return {
        "modelo": modelo.nombre,
        "ancho": int(img.shape[1]),
        "alto": int(img.shape[0]),
        "inferencia_ms": round((time.perf_counter() - t0) * 1000, 1),
        "personas": personas,
    }


@app.get("/health")
def health():
    return {"estado": "ok", "modelo": modelo.nombre, "dispositivo": str(modelo.dispositivo)}


@app.post("/predict")
async def predict(request: Request, x_api_token: str = Header(default=None)):
    if x_api_token != API_TOKEN:
        raise HTTPException(status_code=401, detail="Token invalido")
    datos = await request.body()
    try:
        return await run_in_threadpool(inferir_jpeg, datos)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))


@app.websocket("/ws")
async def ws(websocket: WebSocket):
    if websocket.query_params.get("token") != API_TOKEN:
        await websocket.close(code=1008)
        return
    await websocket.accept()
    try:
        while True:
            datos = await websocket.receive_bytes()
            try:
                res = await run_in_threadpool(inferir_jpeg, datos)
            except Exception as e:
                res = {"error": str(e)}
            await websocket.send_text(json.dumps(res))
    except WebSocketDisconnect:
        pass

## 6. Iniciar el servidor y el tunel

El servidor corre en un hilo y el tunel en un proceso aparte. Al terminar, la celda imprime la URL
publica y el comando exacto para el cliente.

In [ ]:
import uvicorn, subprocess, re

# ---- Servidor ----
if "servidor" in globals() and servidor.started and not servidor.should_exit:
    print("El servidor ya esta corriendo.")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=PUERTO, log_level="warning",
                            ws_ping_interval=20, ws_ping_timeout=20)
    servidor = uvicorn.Server(config)
    hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
    hilo_servidor.start()
    for _ in range(50):
        if servidor.started:
            break
        time.sleep(0.1)
    print("Servidor iniciado:", servidor.started)

# ---- Tunel ----
if "URL_PUBLICA" in globals() and URL_PUBLICA:
    print("El tunel ya esta activo.")
elif TUNEL == "cloudflare":
    URL_PUBLICA = None
    proceso_tunel = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PUERTO}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log_tunel = []

    def _leer_tunel():
        # Se sigue leyendo la salida para que el proceso no se bloquee.
        for linea in proceso_tunel.stdout:
            log_tunel.append(linea)

    threading.Thread(target=_leer_tunel, daemon=True).start()
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", "".join(log_tunel))
        if m:
            URL_PUBLICA = m.group(0)
            break
        time.sleep(0.5)
    if URL_PUBLICA is None:
        print("No se obtuvo la URL del tunel. Ultimas lineas del log:")
        print("".join(log_tunel[-15:]))
    else:
        time.sleep(3)  # el tunel tarda unos segundos en quedar operativo
# ---- ALTERNAR: tunel ngrok ----
# elif TUNEL == "ngrok":
#     from pyngrok import ngrok
#     ngrok.set_auth_token(NGROK_AUTHTOKEN)
#     URL_PUBLICA = ngrok.connect(PUERTO, "http").public_url

if URL_PUBLICA:
    print()
    print("URL publica:", URL_PUBLICA)
    print()
    print("Comando para tu computadora:")
    print(f"python cliente_local.py --url {URL_PUBLICA} --token {API_TOKEN}")

## 7. Prueba rapida (local y a traves del tunel)

In [ ]:
import urllib.request
import matplotlib.pyplot as plt

URL_IMAGEN = "http://images.cocodataset.org/val2017/000000439715.jpg"  # imagen de ejemplo de COCO
datos = urllib.request.urlopen(URL_IMAGEN, timeout=30).read()

req = urllib.request.Request(f"http://localhost:{PUERTO}/predict", data=datos,
                             headers={"Content-Type": "image/jpeg", "X-API-Token": API_TOKEN})
res = json.loads(urllib.request.urlopen(req, timeout=60).read())
print(f"{res['modelo']} | {res['inferencia_ms']} ms | personas: {len(res['personas'])}")

ESQUELETO = [(5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12), (11, 12),
             (11, 13), (13, 15), (12, 14), (14, 16), (0, 1), (0, 2), (1, 3), (2, 4)]
img = cv2.imdecode(np.frombuffer(datos, np.uint8), cv2.IMREAD_COLOR)
for p in res["personas"]:
    x1, y1, x2, y2 = map(int, p["caja"])
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    kp = p["keypoints"]
    for a, b in ESQUELETO:
        if kp[a][2] > 0.5 and kp[b][2] > 0.5:
            cv2.line(img, tuple(map(int, kp[a][:2])), tuple(map(int, kp[b][:2])), (0, 200, 255), 2)
plt.figure(figsize=(10, 7)); plt.imshow(img[:, :, ::-1]); plt.axis("off"); plt.show()

# Verificacion a traves del tunel publico
try:
    salud = json.loads(urllib.request.urlopen(f"{URL_PUBLICA}/health", timeout=30).read())
    print("Tunel OK:", salud)
except Exception as e:
    print("El tunel aun no responde (espera unos segundos y repite):", e)

## 8. Detener el servidor

No ejecutes esta celda mientras uses el cliente. Si usas *Ejecutar todo*, deja `DETENER = False`.

In [ ]:
DETENER = False   # cambia a True y ejecuta esta celda cuando quieras apagar el servidor
if DETENER:
    servidor.should_exit = True
    hilo_servidor.join(timeout=5)
    if "proceso_tunel" in globals():
        proceso_tunel.terminate()
    print("Servidor y tunel detenidos.")
else:
    print("El servidor sigue activo. Pon DETENER = True para apagarlo.")